# 4.3

<div class="alert alert-info"> VAE Imputation: Evaluation

## Ziel
Bewertung der Imputationsgüte. 
Es wird verglichen, wie gut die vom VAE rekonstruierten Werte ("Imputed") mit den tatsächlichen Werten ("Original") übereinstimmen.

## Metriken
- **RMSE**: Root Mean Squared Error (Niedriger ist besser)
- **Verteilungs-Check**: Visueller Vergleich der Dichtefunktionen (KDE).

## Output
- Grafiken zur Imputationsqualität pro Feature.
- Zusammenfassung der Fehlerstatistik.
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

In [ ]:
# ------------------------- Evaluation (PDF Report) -------------------------
import time
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.backends.backend_pdf import PdfPages

sns.set_theme(style="whitegrid")

base_dir = Path.cwd()
inference_root = base_dir.parent / "4.2_Inference" / "Inference_Results"
def log(msg): print(msg)
log(f"Suche Ergebnisse in: {inference_root}")

# ------------------------- Warten auf Ordnererstellung -------------------------
results_folder = None
wait_counter = 0
max_wait_seconds = 1800 

while results_folder is None:
    if not inference_root.exists():
        log("Inferenzordner fehlt noch")
        time.sleep(2)
        continue
        
    timestamp_folders = [f for f in inference_root.iterdir() if f.is_dir()]
    if timestamp_folders:
        candidate = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
        age = time.time() - candidate.stat().st_mtime
        
        if age < 7200:
             results_folder = candidate
             log(f"Nutze Ergebnisse aus: {results_folder.name}")
             break
        else:
             pass
             
    time.sleep(5)
    wait_counter += 5
    if wait_counter >= max_wait_seconds:
        log("Timeout!")
        raise TimeoutError("Keine neuen Ergebnisse von 4.2 gefunden.")

# ------------------------- Setup Loop -------------------------
signal_file_path = results_folder.parent.parent.parent / "4.1_VAE_Imputation" / "Models" / results_folder.name / "DONE_TRAINING"
out_dir_eval = base_dir / "Evaluation_Results" / results_folder.name
out_dir_eval.mkdir(parents=True, exist_ok=True)
log(f"Ergebnis gespeichert in: {out_dir_eval}")

processed_files = set()
summary_stats = []
all_metrics = []
done_training = False

log("\nStarte Monitoring Loop...")

while True:
    
    current_files = list(results_folder.glob("Imputation_Results_*.csv"))
    # ------------------------- Valide Dateien herausfiltern -------------------------
    valid_files = [f for f in current_files if f.stat().st_size > 0]
    new_files = [f for f in valid_files if f.name not in processed_files]

    def get_file_index(p):
        name = p.stem
        parts = name.split("_")
        for part in parts:
             if part.isdigit(): return int(part)
        return 99999
        
    new_files.sort(key=get_file_index)
    
    for file_path in new_files:
        clean_name = file_path.stem.replace("Imputation_Results_Model_", "").replace("Imputation_Results_Run_", "")
        if "Trennlinie" in clean_name:
             processed_files.add(file_path.name)
             continue

        log(f"Evaluation: {clean_name}")
        
        pdf_path = out_dir_eval / f"Analysis_Run_{clean_name}.pdf"
        
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                processed_files.add(file_path.name)
                continue
                
            features = df['Feature'].unique()
            rmse_per_feat = {}
            
            with PdfPages(pdf_path) as pdf:
                # ------------------------- Titelseite -------------------------
                fig_title = plt.figure(figsize=(10, 6))
                plt.axis('off')
                plt.text(0.5, 0.5, f"Evaluation Report\nRun: {clean_name}", ha='center', va='center', fontsize=24)
                pdf.savefig(fig_title)
                plt.close()
                
                for feature in features:
                    subset = df[df['Feature'] == feature]
                    y_true = subset['Original']
                    y_pred = subset['Imputed']
                    if len(y_true) < 2: continue
                    mse = mean_squared_error(y_true, y_pred)
                    rmse = np.sqrt(mse)
                    r2 = r2_score(y_true, y_pred)
                    rmse_per_feat[feature] = rmse
                    all_metrics.append({'Run_ID': clean_name, 'Feature': feature, 'R2': r2, 'RMSE': rmse})
                    
                    # ------------------------- Plot -------------------------
                    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
                    
                    # ------------------------- Scatter -------------------------
                    sns.scatterplot(x=y_true, y=y_pred, ax=ax[0], alpha=0.6, hue=subset['Split'], palette={'Train': 'blue', 'Test': 'red'})
                    low = min(y_true.min(), y_pred.min())
                    high = max(y_true.max(), y_pred.max())
                    ax[0].plot([low, high], [low, high], 'r--', label='Ideal')
                    ax[0].set_title(f"Scatter Plot (R²={r2:.3f})")
                    ax[0].legend()
                    
                    # ------------------------- KDE -------------------------
                    sns.kdeplot(y_true, ax=ax[1], label="Original", fill=True, alpha=0.3)
                    sns.kdeplot(y_pred, ax=ax[1], label="Imputed", fill=True, alpha=0.3)
                    ax[1].set_title(f"Distribution (RMSE={rmse:.3f})")
                    ax[1].legend()
                    
                    plt.suptitle(f"Feature: {feature}", fontsize=16)
                    plt.tight_layout()
                    
                    pdf.savefig(fig)
                    plt.close()
                
            avg_rmse = np.mean(list(rmse_per_feat.values())) if rmse_per_feat else 0
            summary_stats.append({
                "Run": clean_name,
                "Avg_RMSE": avg_rmse,
                "Features": list(rmse_per_feat.keys())
            })
            processed_files.add(file_path.name)
            log(f"Erstelle PDF: {pdf_path.name}")
            
        except Exception as e:
            log(f"Fehler bei {file_path.name}: {e}")
            processed_files.add(file_path.name)

    if signal_file_path.exists():
        if not done_training:
             log("Signal DONE_TRAINING erkannt.")
             done_training = True
             
        if not new_files:
             current_check = [f for f in list(results_folder.glob("Imputation_Results_*.csv")) if f.stat().st_size > 0]
             remaining = [f for f in current_check if f.name not in processed_files]
             if not remaining:
                 # ------------------------- Globales .pdf-Ergebnis -------------------------
                 if summary_stats:
                    df_summary = pd.DataFrame(summary_stats)
                    csv_path = out_dir_eval / "Summary_Evaluation.csv"
                    df_summary.to_csv(csv_path, index=False)
                    
                    # ------------------------- .pdf Zusammenfassung -------------------------
                    sum_pdf_path = out_dir_eval / "Evaluation_Summary_Report.pdf"
                    with PdfPages(sum_pdf_path) as pdf:
                        # ------------------------- Tabelle -------------------------
                        fig, ax = plt.subplots(figsize=(12, len(df_summary)*0.3 + 2))
                        ax.axis('tight')
                        ax.axis('off')
                        ax.table(cellText=df_summary[['Run', 'Avg_RMSE']].values, colLabels=['Run', 'Avg_RMSE'], loc='center')
                        ax.set_title("Overall Performance Summary", fontsize=16)
                        pdf.savefig(fig)
                        plt.close()
                        
                        # ------------------------- Balkendiagramm erstellen -------------------------
                        fig2, ax2 = plt.subplots(figsize=(12, 6))
                        sns.barplot(data=df_summary, x='Run', y='Avg_RMSE', ax=ax2)
                        plt.xticks(rotation=90)
                        plt.tight_layout()
                        pdf.savefig(fig2)
                        plt.close()
                        
                    log(f"Global Summary PDF created: {sum_pdf_path.name}")

                 log("Alle Evaluierungen abgeschlossen.")
                 break
                 
    time.sleep(5)

Monitoring models from: 2026-01-10_01-27-36
Monitoring results in: 2026-01-10_01-27-36
Saving Evaluation to: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Evaluation\Evaluation_Results\2026-01-10_01-27-36
Starte Evaluation Monitoring...
.

.

.

.

.

...

.

Evaluated Imputation_Results_Run_030.csv


--> PDF Report for Run_030 created.

DONE_INFERENCE erkannt. Beende Evaluation.

Zusammenfassung gespeichert: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Evaluation\Evaluation_Results\2026-01-10_01-27-36\Evaluation_Summary.csv
Run_ID
Run_021    0.583395
Run_006    0.581795
Run_007    0.581245
Run_014    0.575183
Run_005    0.560215
Name: R2, dtype: float64
Generating PDF Report: Evaluation_Report.pdf...


C:\Users\lucca\AppData\Local\Temp\ipykernel_12020\2606209930.py:207: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45)


PDF Saved.


In [ ]:
# ------------------------- Finalen Bericht sichern -------------------------
if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    summary_path = eval_output_dir / "Evaluation_Summary.csv"
    df_metrics.to_csv(summary_path, index=False)
    print(f"\nZusammenfassung gespeichert: {summary_path}")
    
    # ------------------------- Top 5 anzeigen -------------------------
    print(df_metrics.groupby('Run_ID')['R2'].mean().sort_values(ascending=False).head())

    # ------------------------- PDF Bericht Erstellung -------------------------
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    pdf_path = eval_output_dir / "Evaluation_Report.pdf"
    print(f"Generating PDF Report: {pdf_path.name}...")
    
    with PdfPages(pdf_path) as pdf:

        # ------------------------- PDF Bericht Erstellung -------------------------
        fig, ax = plt.subplots(figsize=(11.69, 8.27)) # A4
        ax.axis('off')
        ax.text(0.5, 0.95, "VAE Imputation - Evaluation Report", ha='center', fontsize=24, weight='bold')
        ax.text(0.5, 0.90, f"Date: {time.strftime('%Y-%m-%d %H:%M')}", ha='center', fontsize=12)
        
        # ------------------------- Top 10 Modelle -------------------------
        agg_funcs = {'R2': 'mean', 'RMSE': 'mean', 'Features_List': 'first'}
        top_n = df_metrics.groupby('Run_ID').agg(agg_funcs).sort_values('R2', ascending=False).head(10).reset_index()
        top_n['R2'] = top_n['R2'].round(4)
        top_n['RMSE'] = top_n['RMSE'].round(4)
        
        table_data = [top_n.columns.values.tolist()] + top_n.values.tolist()
        table = ax.table(cellText=table_data, colLabels=top_n.columns, loc='center', cellLoc='center', bbox=[0.1, 0.4, 0.8, 0.4])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.2)
        ax.text(0.5, 0.82, "Top 10 Models (by Mean R²)", ha='center', fontsize=16)
        
        pdf.savefig(fig)
        plt.close()
        
        # ------------------------- Perfomanz Ergebnisse -------------------------
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        sns.boxplot(data=df_metrics, x='Feature', y='R2', ax=ax)
        ax.set_title("R² Score Distribution per Feature across all Runs", fontsize=16)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
        ax.set_ylim(-0.1, 1.1)
        pdf.savefig(fig)
        plt.close()
        
        # ------------------------- Details zum besten Modell -------------------------
        best_run_id = top_n.iloc[0]['Run_ID']
        best_run_file = results_root / f"Imputation_Results_{best_run_id}.csv"
        
        if best_run_file.exists():
            df_best = pd.read_csv(best_run_file)
            features = df_best['Feature'].unique()
            
            # ------------------------- Scatter Plots für Modell -------------------------
            cols = 3
            rows = (len(features) + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(11.69, 4*rows))
            fig.suptitle(f"Best Model: {best_run_id} - Original vs Imputed", fontsize=16)
            axes = axes.flatten()
            
            for i, feat in enumerate(features):
                ax = axes[i]
                subset = df_best[df_best['Feature'] == feat]
                
                # ------------------------- Scatter -------------------------
                sns.scatterplot(data=subset, x='Original', y='Imputed', ax=ax, alpha=0.3, s=10)
                
                # ------------------------- Ideallinie [Original = Imputed] -------------------------
                min_val = min(subset['Original'].min(), subset['Imputed'].min())
                max_val = max(subset['Original'].max(), subset['Imputed'].max())
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5)
                
                # ------------------------- Metrik -------------------------
    
                r2_vals = df_metrics[(df_metrics['Run_ID'] == best_run_id) & (df_metrics['Feature'] == feat)]['R2'].values
                r2_val = r2_vals[0] if len(r2_vals) > 0 else 0.0

                ax.set_title(f"{feat} (R²: {r2_val:.2f})")
                
            for j in range(i+1, len(axes)): axes[j].axis('off')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            pdf.savefig(fig)
            plt.close()
            
    print(f"PDF Saved.")